# TransitPulse Ridership Forecasting

This notebook trains a Prophet model on daily total ridership and forecasts the next 90 days.

In [ ]:
from pathlib import Path
from os import getenv
from urllib.parse import quote_plus
import pickle

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from prophet import Prophet
from sqlalchemy import create_engine, text

plt.rcParams["figure.figsize"] = (14, 7)
plt.rcParams["axes.titlesize"] = 16
plt.rcParams["axes.labelsize"] = 12

cwd = Path.cwd()
env_path = cwd / ".env"
if not env_path.exists():
    env_path = cwd.parent / ".env"

if not env_path.exists():
    raise FileNotFoundError("Could not find .env in the current directory or parent directory.")

load_dotenv(env_path)
PROJECT_ROOT = env_path.parent
CHART_DIR = PROJECT_ROOT / "notebooks" / "charts"
MODEL_DIR = PROJECT_ROOT / "models"
CHART_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

required_vars = ["DB_HOST", "DB_PORT", "DB_NAME", "DB_USER", "DB_PASSWORD"]
missing = [name for name in required_vars if not getenv(name)]
if missing:
    raise RuntimeError(f"Missing required .env variables: {', '.join(missing)}")

user = quote_plus(getenv("DB_USER", ""))
password = quote_plus(getenv("DB_PASSWORD", ""))
host = getenv("DB_HOST")
port = getenv("DB_PORT")
database = getenv("DB_NAME")
connection_string = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}"
engine = create_engine(connection_string)

print(f"Forecast chart will be saved to: {CHART_DIR / 'forecast.png'}")
print(f"Model will be saved to: {MODEL_DIR / 'prophet_ridership.pkl'}")

## Query Daily Ridership

In [ ]:
daily_ridership = pd.read_sql(
    text(
        """
        SELECT
            d.full_date AS ds,
            SUM(f.passenger_count) AS y
        FROM fact_ridership f
        JOIN dim_date d ON f.date_id = d.date_id
        GROUP BY d.full_date
        ORDER BY d.full_date
        """
    ),
    engine,
)

daily_ridership["ds"] = pd.to_datetime(daily_ridership["ds"])
daily_ridership["y"] = pd.to_numeric(daily_ridership["y"])

if len(daily_ridership) < 2:
    raise RuntimeError("Prophet needs at least two daily ridership observations to train.")

daily_ridership.head()

## Train Prophet Model

In [ ]:
model = Prophet(
    interval_width=0.95,
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
)
model.fit(daily_ridership)

future = model.make_future_dataframe(periods=90, freq="D")
forecast = model.predict(future)
forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].tail()

## Save Forecast Chart

In [ ]:
fig = model.plot(forecast, xlabel="Date", ylabel="Daily Ridership")
ax = fig.gca()
ax.set_title("Daily Ridership Forecast - Next 90 Days")
ax.grid(True, alpha=0.3)

forecast_path = CHART_DIR / "forecast.png"
fig.tight_layout()
fig.savefig(forecast_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {forecast_path}")

## Save Trained Model

In [ ]:
model_path = MODEL_DIR / "prophet_ridership.pkl"
with model_path.open("wb") as model_file:
    pickle.dump(model, model_file)

print(f"Saved {model_path}")

In [ ]:
engine.dispose()